In [1]:
data = [{'time_start': '05:54',
  'time_end': '06:13',
  'reason': "Неожиданное повышение номера из-за 'прикольной шляпы' — дружелюбие и юмор, демонстрирующие гостеприимство Ямайки."},
 {'time_start': '07:00',
  'time_end': '07:29',
  'reason': 'Встреча с племянником Боба Марли — удивительный и вдохновляющий момент, вызывающий сильные эмоции.'},
 {'time_start': '08:04',
  'time_end': '08:30',
  'reason': 'Радостная реакция на потрясающий вид из окна — эмоциональный момент, передающий восторг от путешествия.'},
 {'time_start': '04:29',
  'time_end': '04:31',
  'reason': "Шутка о том, что фигура Усейна Болта сделана из 'болтов' — юмористический момент, вызывающий улыбку."},
 {'time_start': '11:04',
  'time_end': '11:11',
  'reason': 'Неожиданная статистика о потреблении травки на Ямайке — развенчание стереотипов, вызывающее интерес.'},
 {'time_start': '10:01',
  'time_end': '10:08',
  'reason': 'Юмористическая беседа о стереотипах и привлекательности — вызывает улыбку и легко воспринимается зрителями.'},
 {'time_start': '13:13',
  'time_end': '13:30',
  'reason': 'Знакомство с настоящими растаманами в их деревне — уникальный и познавательный момент, погружающий в культуру Ямайки.'}]

In [2]:
import os

os.environ['IMAGEMAGICK_BINARY'] = r'C:\Program Files\ImageMagick-7.1.1-Q16-HDRI\magick.exe'


In [47]:
import whisper
from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip
from moviepy.video.tools.subtitles import SubtitlesClip
import os

# Транскрибирование аудио из видео с помощью Whisper
def transcribe_video(video_path, model="base"):
    # Загружаем модель Whisper
    model = whisper.load_model(model)
    
    # Загружаем видеофайл
    clip = VideoFileClip(video_path)
    
    # Извлекаем аудио из видео
    audio_path = "temp_audio.wav"
    clip.audio.write_audiofile(audio_path, codec='pcm_s16le')
    
    # Транскрибируем аудио
    result = model.transcribe(audio_path)
    os.remove(audio_path)  # Удаляем временный аудиофайл
    
    return result['segments']

# Путь к файлу шрифта
font_path = "Obelix Pro.ttf"

# Разбивает текст на фрагменты, проверяя ширину текста
def split_text_to_chunks(text, clip):
    words = text.split()
    chunks = []
    current_chunk = []
    max_width = clip.w - 20  # Оставим 10 пикселей с каждой стороны для отступа

    for word in words:
        current_chunk.append(word)
        # Создаем временный TextClip, чтобы проверить ширину
        temp_clip = TextClip(' '.join(current_chunk), font=font_path, fontsize=40, color="yellow")
        if temp_clip.w > max_width:
            # Если ширина превышает максимальную, убираем последнее слово и добавляем в chunks
            current_chunk.pop()  # Убираем последнее слово
            if current_chunk:  # Если есть слова для добавления
                chunks.append(' '.join(current_chunk))
            current_chunk = [word]  # Начинаем новый chunk с текущим словом

    # Добавляем последний chunk, если он не пуст
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Генерация субтитров
def create_subtitles(video_path, segments, output_path):
    # Загружаем видео
    clip = VideoFileClip(video_path)

    subtitle_clips = []
    
    for segment in segments:
        # Разбиваем текст на фрагменты
        chunks = split_text_to_chunks(segment['text'].strip(), clip)
        
        # Определяем длительность каждого фрагмента
        chunk_duration = (segment['end'] - segment['start']) / len(chunks)
        
        # Создаем отдельный TextClip для каждого фрагмента
        for i, chunk in enumerate(chunks):
            chunk_start = segment['start'] + i * chunk_duration
            
            # Создаем TextClip с нужной стилизацией
            text_clip = TextClip(chunk, font=font_path, fontsize=40, color="yellow",
                                 stroke_color="black", stroke_width=4, size=(clip.w, None))
            text_clip = text_clip.set_duration(chunk_duration).set_start(chunk_start)
            subtitle_clips.append(text_clip)

    # Накладываем субтитры на видео, поднимаем их чуть выше
    video_with_subtitles = CompositeVideoClip([clip] + [sub.set_pos(('center', clip.h * 0.8)) for sub in subtitle_clips])
    
    # Сохраняем результат
    video_with_subtitles.write_videofile(output_path, codec="libx264", fps=clip.fps)

video_path = "output_video.mp4"  # путь к входному видео
output_path = "output_video_with_subtitles_final.mp4"  # путь для сохранения видео с субтитрами

def process_video(video_path, output_path, model="base"):
    print("Транскрибируем видео...")
    segments = transcribe_video(video_path, model=model)
    
    print("Накладываем субтитры...")
    create_subtitles(video_path, segments, output_path)
    
    print(f"Видео сохранено с субтитрами: {output_path}")

In [48]:
process_video(video_path, output_path)

Транскрибируем видео...
MoviePy - Writing audio in temp_audio.wav


MoviePy - Done.
Накладываем субтитры...
Moviepy - Building video output_video_with_subtitles_final.mp4.
MoviePy - Writing audio in output_video_with_subtitles_finalTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video output_video_with_subtitles_final.mp4



Moviepy - Done !
Moviepy - video ready output_video_with_subtitles_final.mp4
Видео сохранено с субтитрами: output_video_with_subtitles_final.mp4


In [45]:
from moviepy.editor import VideoFileClip, CompositeVideoClip, TextClip
from moviepy.video.tools.subtitles import SubtitlesClip


# Путь к файлу шрифта
font_path = "Obelix Pro.ttf"

# Разбивает текст на фрагменты, проверяя ширину текста
def split_text_to_chunks(text, clip):
    words = text.split()
    chunks = []
    current_chunk = []
    max_width = clip.w - 20  # Оставим 10 пикселей с каждой стороны для отступа

    for word in words:
        current_chunk.append(word)
        # Создаем временный TextClip, чтобы проверить ширину
        temp_clip = TextClip(' '.join(current_chunk), font=font_path, fontsize=40, color="red")
        if temp_clip.w > max_width:
            # Если ширина превышает максимальную, убираем последнее слово и добавляем в chunks
            current_chunk.pop()  # Убираем последнее слово
            if current_chunk:  # Если есть слова для добавления
                chunks.append(' '.join(current_chunk))
            current_chunk = [word]  # Начинаем новый chunk с текущим словом

    # Добавляем последний chunk, если он не пуст
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

# Генерация субтитров
def create_subtitles(video_path, segments, output_path):
    # Загружаем видео
    clip = VideoFileClip(video_path)

    subtitle_data = []
    
    for segment in segments:
        # Разбиваем текст на фрагменты
        chunks = split_text_to_chunks(segment['text'].strip(), clip)
        
        # Определяем длительность каждого фрагмента
        chunk_duration = (segment['end'] - segment['start']) / len(chunks)
        
        # Создаем данные для SubtitlesClip
        for i, chunk in enumerate(chunks):
            chunk_start = segment['start'] + i * chunk_duration
            chunk_end = chunk_start + chunk_duration
            # Добавляем только (начало, конец, текст)
            subtitle_data.append(((chunk_start, chunk_end), chunk))  # Изменено на кортеж из двух элементов

    # Создаем SubtitlesClip
    subtitles = SubtitlesClip(subtitle_data, make_textclip=lambda txt: TextClip(txt, font=font_path, fontsize=40, color="red",
                                                                             stroke_color="black", stroke_width=4))

    # Накладываем субтитры на видео, поднимаем их чуть выше
    video_with_subtitles = CompositeVideoClip([clip, subtitles.set_pos(('center', clip.h * 0.8))])
    
    # Сохраняем результат
    video_with_subtitles.write_videofile(output_path, codec="libx264", fps=clip.fps)


In [ ]:
segments = transcribe_video(video_path, model='base')

In [46]:
# Пример использования
video_path = "output_video.mp4"  # путь к входному видео
output_path = "output_video_with_subtitles.mp4"  # путь для сохранения видео с субтитрами

create_subtitles(video_path, segments, output_path)
    
print(f"Видео сохранено с субтитрами: {output_path}")





Moviepy - Building video output_video_with_subtitles.mp4.
MoviePy - Writing audio in output_video_with_subtitlesTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video output_video_with_subtitles.mp4



Moviepy - Done !
Moviepy - video ready output_video_with_subtitles.mp4
Видео сохранено с субтитрами: output_video_with_subtitles.mp4
